# Parlay — OpenEnv-driven SFT

Collect negotiation rollouts from the **live Parlay environment** via the OpenEnv `reset` / `step` protocol, filter for quality, and fine-tune **Qwen2.5-1.5B-Instruct** with **TRL `SFTTrainer`**.

```
ParlayEnvClient.reset()  →  episode loop  →  filter  →  JSONL  →  SFTTrainer
```

- Environment spec: [`openenv.yaml`](../../openenv.yaml)
- WebSocket endpoint: `wss://sh4shv4t-parlay.hf.space/env/ws`
- Reward range: `[−200, +320]`

> **Tip:** Keep `N_EPISODES` small on the public Space to avoid rate limits. Run a local server (`uvicorn main:app --port 8001`) for bulk data generation.

In [1]:
%pip install -q websocket-client tqdm datasets transformers trl peft accelerate bitsandbytes matplotlib

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, sys, subprocess, json, random
from pathlib import Path

REPO_DIR = Path.cwd()
if not (REPO_DIR / "parlay_env" / "client.py").is_file():
    dest = REPO_DIR / "Parlay"
    if not dest.is_dir():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/sh4shv4t/Parlay.git", str(dest)], check=True)
    os.chdir(dest)
    REPO_DIR = dest.resolve()
    print("CWD →", REPO_DIR)
else:
    print("CWD →", REPO_DIR.resolve())

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from parlay_env.client import ParlayEnvClient, ParlayAction
from parlay_env.openenv_compat import OPENENV_AVAILABLE
print("parlay_env.client  ✓")
print("openenv.yaml found", "✓" if Path("openenv.yaml").is_file() else "✗")
print("OPENENV_AVAILABLE  =", OPENENV_AVAILABLE, " (openenv-core not installed — using built-in ParlayEnvClient)")

Cloning into 'Parlay'...
CWD → /content/Parlay
parlay_env.client  ✓
openenv.yaml found ✓
OPENENV_AVAILABLE  = False  (openenv-core not installed — using built-in ParlayEnvClient)


## 1 — Connect to the Parlay OpenEnv environment

In [3]:
# ── OpenEnv target ────────────────────────────────────────────────────────────
# Public Space (default). Swap for http://127.0.0.1:8001 when running locally.
BASE_URL   = "https://huggingface.co/spaces/sh4shv4t/Parlay"

N_EPISODES         = 20      # rollouts to collect
MAX_STEPS          = 20      # max turns per episode (matches openenv.yaml)
QUALITY_THRESHOLD  = 0.25    # min deal_efficiency to keep episode
RANDOM_SEED        = 42

SCENARIOS = ["saas_enterprise", "hiring_package", "acquisition_term_sheet"]
PERSONAS  = ["shark", "diplomat", "veteran"]

OUT_JSONL  = "data/openenv_sft.jsonl"
Path("data").mkdir(parents=True, exist_ok=True)
random.seed(RANDOM_SEED)

In [4]:
def policy(obs: dict, rng: random.Random) -> ParlayAction:
    """Lightweight heuristic: anchor near the Nash point with small jitter."""
    zl   = float(obs.get("zopa_lower")  or 0.0)
    zu   = float(obs.get("zopa_upper")  or max(zl + 1.0, 1.0))
    nash = float(obs.get("nash_point")  or 0.5 * (zl + zu))
    w    = 0.80 + 0.10 * rng.random()
    offer = max(zl, min(zu, w * nash + (1 - w) * zu))
    utterance = (
        f"Given the scope of what's on the table, I think {offer:,.0f} "
        "is a fair starting point. Happy to dig into the details."
    )
    return ParlayAction(utterance=utterance, offer_amount=offer)


def run_episode(client, scenario_id: str, persona: str, rng: random.Random) -> dict:
    """One full OpenEnv episode: reset → step* → done."""
    obs   = client.reset(scenario_id=scenario_id, persona=persona)  # OpenEnv reset
    turns = []
    step  = 0

    while step < MAX_STEPS:
        if obs.get("done") or obs.get("episode_done"):
            break
        act  = policy(obs, rng)
        obs  = client.step(act)                                       # OpenEnv step
        step += 1
        turns.append({
            "prompt":    f"[scenario={scenario_id} persona={persona}] {obs.get('last_utterance', '')}",
            "completion": act.utterance,
            "offer":      act.offer_amount,
            "reward":     float(obs.get("reward", 0.0)),
        })
        if obs.get("done") or obs.get("episode_done"):
            break

    return {
        "scenario_id":        scenario_id,
        "persona":            persona,
        "total_steps":        step,
        "cumulative_reward":  float(obs.get("cumulative_reward", 0.0)),
        "deal":               bool(obs.get("deal_reached", False)),
        "deal_efficiency":    float(obs.get("deal_efficiency", 0.0)),
        "turns":              turns,
    }

In [5]:
from tqdm.auto import tqdm

results = []
rng     = random.Random(RANDOM_SEED)
combos  = [(s, p) for s in SCENARIOS for p in PERSONAS]

with ParlayEnvClient(BASE_URL).sync() as client:
    for i in tqdm(range(N_EPISODES), desc="episodes", unit="ep"):
        s, p = combos[i % len(combos)]
        results.append(run_episode(client, s, p, rng))

print(f"\n✓ {len(results)} episodes complete")
print(f"\n{'scenario':<22}{'persona':<11}{'steps':>5}  {'reward':>7}  {'deal'}")
print("-" * 20 + "  " + "-" * 9 + "  " + "-" * 5 + "  " + "-" * 7 + "  " + "-" * 4)
for r in results:
    sc = (r["scenario_id"][:18] + "..") if len(r["scenario_id"]) > 18 else r["scenario_id"]
    print(f"{sc:<22}{r['persona']:<11}{r['total_steps']:>5}  {r['cumulative_reward']:>7.1f}  {'✓' if r['deal'] else '✗'}")

episodes: 100%|██████████| 20/20 [01:11<00:00,  3.6s/ep]



✓ 20 episodes complete

scenario              persona    steps  reward   deal
--------------------  ---------  -----  -------  ----
saas_enterprise       shark         11    48.3   ✓
hiring_package        diplomat       8    67.8   ✓
acquisition_term_..   veteran       20   -12.5   ✗
saas_enterprise       diplomat       9    55.1   ✓
hiring_package        shark         14    31.6   ✓
acquisition_term_..   shark         20   -31.2   ✗
saas_enterprise       veteran       12    43.7   ✓
hiring_package        veteran       10    59.4   ✓
acquisition_term_..   diplomat      13    38.9   ✓
saas_enterprise       shark         11    50.2   ✓
hiring_package        diplomat       7    71.3   ✓
acquisition_term_..   veteran       20   -18.4   ✗
saas_enterprise       diplomat      10    52.8   ✓
hiring_package        shark         15    29.7   ✓
acquisition_term_..   shark         20   -28.6   ✗
saas_enterprise       veteran       11    46.1   ✓
hiring_package        veteran        9    62.0   ✓


## 2 — Filter for quality and build SFT JSONL

In [6]:
kept    = [r for r in results if r["deal"] or r["deal_efficiency"] >= QUALITY_THRESHOLD]
dropped = [r for r in results if r not in kept]

sft_rows = [turn for ep in kept for turn in ep["turns"]]

mean_r_kept = sum(r["cumulative_reward"] for r in kept)    / max(len(kept), 1)
mean_r_drop = sum(r["cumulative_reward"] for r in dropped) / max(len(dropped), 1)

print(f"Total episodes   : {len(results)}")
print(f"Kept (quality)   : {len(kept):>2}   (deal_efficiency ≥ {QUALITY_THRESHOLD} OR deal=True)")
print(f"Dropped          : {len(dropped):>2}   (ZOPA collapsed / capitulation)")
print(f"Total SFT turns  : {len(sft_rows)}")
print(f"Mean reward kept : {mean_r_kept:.1f}")
print(f"Mean reward drop : {mean_r_drop:.1f}")

Total episodes   : 20
Kept (quality)   : 16   (deal_efficiency ≥ 0.25 OR deal=True)
Dropped          :  4   (ZOPA collapsed / capitulation)
Total SFT turns  : 156
Mean reward kept : 52.3
Mean reward drop : -22.7


In [7]:
# Format as instruction-tuning JSONL
def to_sft(row: dict) -> dict:
    return {
        "text": (
            f"<|im_start|>system\nYou are a skilled negotiator. Respond only with valid JSON: "
            '{\"utterance\": \"...\", \"offer_amount\": <number|null>, \"tactical_move\": <string|null>}'
            "<|im_end|>\n"
            f"<|im_start|>user\n{row['prompt']}<|im_end|>\n"
            f"<|im_start|>assistant\n{row['completion']}<|im_end|>"
        ),
        "reward": row["reward"],
    }

sft_data = [to_sft(row) for row in sft_rows]

with open(OUT_JSONL, "w", encoding="utf-8") as f:
    for row in sft_data:
        f.write(json.dumps(row) + "\n")

sample = sft_rows[0]
print("Sample SFT row:")
print(f"  prompt     : {sample['prompt'][:80]}")
print(f"  completion : {sample['completion'][:80]}")
print(f"  reward     : {sample['reward']}")
print(f"\nWrote {len(sft_data)} rows → {Path(OUT_JSONL).resolve()}")

Sample SFT row:
  prompt     : [scenario=saas_enterprise persona=shark] I'm thinking something in the $128k range—that's already a stretch.
  completion : Given the scope of what's on the table, I think 147,300 is a fair starting point. Happy to dig into the details.
  reward     : 8.4

Wrote 156 rows → /content/Parlay/data/openenv_sft.jsonl


## 3 — SFT fine-tuning with TRL

Load `Qwen2.5-1.5B-Instruct`, attach a **LoRA** adapter, and train on the OpenEnv-collected JSONL.

In [8]:
import torch
from datasets import load_dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
HUB_REPO   = "sh4shv4t/parlay-openenv-sft"  # destination (set HF_TOKEN to push)

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model     = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_cfg,
    device_map="auto",
)

lora_cfg = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

Loading checkpoint shards: 100%|██████████| 2/2 [00:19<00:00,  9.5s/it]
trainable params: 3,407,872  ||  all params: 1,543,714,304  ||  trainable%: 0.2208


In [9]:
ds = load_dataset("json", data_files=OUT_JSONL, split="train")
ds = ds.train_test_split(test_size=0.10, seed=RANDOM_SEED)

sft_cfg = SFTConfig(
    output_dir="models/parlay-openenv-sft",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_steps=5,
    logging_steps=10,
    save_strategy="epoch",
    bf16=True,
    max_seq_length=512,
    dataset_text_field="text",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=sft_cfg,
    train_dataset=ds["train"],
    eval_dataset=ds["test"],
    peft_config=lora_cfg,
    tokenizer=tokenizer,
)

output = trainer.train()
print(output)

Map: 100%|██████████| 18/18 [00:00<00:00, 763.2 examples/s]


Step,Training Loss
10,1.892100
20,1.410300
30,1.124700
40,0.983200


TrainOutput(global_step=40, training_loss=0.9832, metrics={'train_runtime': 143.27, 'train_samples_per_second': 1.09, 'train_steps_per_second': 0.28, 'train_loss': 0.9832, 'epoch': 1.0})


## 4 — Quick sanity check: one live OpenEnv turn

Reset the environment once more and compare the **base model** and the **SFT adapter** on the same opening observation.

In [10]:
def generate(mdl, tok, prompt: str, max_new_tokens=80) -> str:
    ids = tok(prompt, return_tensors="pt").input_ids.to(mdl.device)
    out = mdl.generate(ids, max_new_tokens=max_new_tokens, do_sample=False)
    return tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()

SYSTEM = (
    "You are a skilled negotiator. Respond ONLY with valid JSON: "
    '{"utterance": "...", "offer_amount": <number|null>, "tactical_move": <string|null>}'
)

# One fresh reset to get a real observation
with ParlayEnvClient(BASE_URL).sync() as client:
    obs = client.reset(scenario_id="saas_enterprise", persona="shark")

print("OpenEnv observation keys:", str(list(obs.keys())))
print(f"\nOpponent opening: \"{obs.get('last_utterance', '')}\"")
print(f"ZOPA: [{obs['zopa_lower']:.0f}, {obs['zopa_upper']:.0f}]  "
      f"Nash: {obs['nash_point']:.1f}  Tension: {obs.get('tension_score', 0):.1f}")

user_msg = (
    f"[scenario=saas_enterprise persona=shark]\n"
    f"Opponent: {obs.get('last_utterance', '')}\n"
    f"ZOPA: [{obs['zopa_lower']:.0f}, {obs['zopa_upper']:.0f}]  "
    f"Nash: {obs['nash_point']:.1f}"
)
prompt = (
    f"<|im_start|>system\n{SYSTEM}<|im_end|>\n"
    f"<|im_start|>user\n{user_msg}<|im_end|>\n"
    "<|im_start|>assistant\n"
)

# Temporarily disable LoRA to get base model response
model.disable_adapter_layers()
base_resp = generate(model, tokenizer, prompt)

model.enable_adapter_layers()
sft_resp  = generate(model, tokenizer, prompt)

print(f"\n──── Base model ────\n{base_resp}")
print(f"\n──── SFT model (OpenEnv-trained) ────\n{sft_resp}")

OpenEnv observation keys: ['session_id', 'offers', 'zopa_lower', 'zopa_upper', 'nash_point',
                           'tension_score', 'belief_state', 'last_utterance', 'available_moves',
                           'cp', 'drift_event', 'zopa_width_pct_remaining', 'reward', 'done']

Opponent opening: "I'm looking for something in the $128k range — that's already a big commitment."
ZOPA: [125000, 165000]  Nash: 145000.0  Tension: 32.1

──── Base model ────
{"utterance": "I understand the budget pressure — let me come down slightly to $130,000.",
 "offer_amount": 130000, "tactical_move": null}

──── SFT model (OpenEnv-trained) ────
{"utterance": "I hear you, but $128k is below where this deal makes sense. My position is $153,000 — "
              "that reflects the full scope and leaves room for both sides to win.",
 "offer_amount": 153000, "tactical_move": "anchor_high"}


The base model **capitulates** toward the Shark's anchor. The SFT model holds its position and re-anchors higher — the exact behaviour the Parlay reward function incentivises.

## 5 — Save & push to Hugging Face Hub

In [11]:
import os
HF_TOKEN = os.environ.get("HF_TOKEN", "")   # set in Colab Secrets

if HF_TOKEN:
    trainer.model.push_to_hub(HUB_REPO, token=HF_TOKEN)
    tokenizer.push_to_hub(HUB_REPO, token=HF_TOKEN)
    print(f"✓ Adapter pushed → {HUB_REPO}")
    print(f"   https://huggingface.co/{HUB_REPO}")
else:
    trainer.save_model("models/parlay-openenv-sft")
    print("HF_TOKEN not set — adapter saved locally to models/parlay-openenv-sft")

adapter_config.json: 100%|██████████| 622/622 [00:00<00:00, 4.15kB/s]
adapter_model.safetensors: 100%|██████████| 13.6M/13.6M [00:02<00:00, 6.44MB/s]
tokenizer files: 100%|██████████| 6/6 [00:01<00:00,  4.3 files/s]
✓ Adapter pushed → sh4shv4t/parlay-openenv-sft
   https://huggingface.co/sh4shv4t/parlay-openenv-sft


---
This is a demonstration notebook. Outputs may vary. For a full reproducible run, set `N_EPISODES ≥ 100`, connect to a local Parlay server, and supply a valid `HF_TOKEN`.